# Genre2Vec — Data to Embeddings
Reads `songs.csv` and `tags.csv`, trains skip-gram, saves `genre_embeddings.json`.

In [5]:
import numpy as np
import os
import json
from collections import defaultdict

## Config

In [6]:
DATA_DIR = './data/'
CSV_DIR = DATA_DIR + './csv/'
SONGS_PATH = CSV_DIR + "songs.csv"
TAGS_PATH = CSV_DIR + "tags.csv"
OUTPUT_PATH = DATA_DIR + "/embeddings/genre_embeddings.json"

MIN_FREQ = 3
WINDOW = 4
EMBED_DIM = 64
NEG_SAMPLES = 5
EPOCHS = 5
LR = 0.025
BATCH_SIZE = 1024
TAG_POP_CAP = 5

In [7]:
paths = [SONGS_PATH, TAGS_PATH]
for p in paths:
    assert os.path.exists(p), f"Missing file: {p}"

os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)

## 1. Load & Clean Data

In [8]:
def read_csv(path):
    with open(path) as f:
        lines = f.read().strip().splitlines()
    headers = [h.strip('"') for h in lines[0].split(',')]
    rows = []
    for line in lines[1:]:
        parts = [p.strip('"') for p in line.split(';')]
        if len(parts) == len(headers):
            rows.append(dict(zip(headers, parts)))
    return rows

docs = defaultdict(set)
for row in read_csv(SONGS_PATH):
    docs[row["spotify_id"]].add(row["genre_name"])

for row in read_csv(TAGS_PATH):
    weight = min(int(row["popularity"]), TAG_POP_CAP)
    for _ in range(weight):
        docs[row["song_spotify_id"]].add(row["tag"])

documents = [list(tokens) for tokens in docs.values() if len(tokens) >= 2]
print(f"{len(documents):,} songs | {sum(len(d) for d in documents):,} total tokens")

80,912 songs | 1,139,923 total tokens


## 2. Vocabulary

In [10]:
freq = defaultdict(int)
for doc in documents:
    for t in doc: freq[t] += 1

idx2token = [t for t, c in sorted(freq.items()) if c >= MIN_FREQ]
token2idx = {t: i for i, t in enumerate(idx2token)}
V = len(idx2token)
print(f"{V} tokens in vocab")

37640 tokens in vocab


## 3. Skip-gram Pairs

In [11]:
pairs = []
for doc in [[token2idx[t] for t in d if t in token2idx] for d in documents]:
    if len(doc) < 2: continue
    for i, c in enumerate(doc):
        for j in range(max(0, i - WINDOW), min(len(doc), i + WINDOW + 1)):
            if j != i: pairs.append((c, doc[j]))

pairs = np.array(pairs, dtype=np.int32)
print(f"{len(pairs):,} training pairs")

6,580,804 training pairs


## 4. Train Genre2Vec

In [12]:
np.random.seed(42)
Win  = (np.random.randn(V, EMBED_DIM) * 0.01).astype(np.float32)
Wout = (np.random.randn(V, EMBED_DIM) * 0.01).astype(np.float32)

for epoch in range(EPOCHS):
    idx = np.random.permutation(len(pairs))
    C, CTX = pairs[idx, 0], pairs[idx, 1]
    loss = 0.0

    for s in range(0, len(C), BATCH_SIZE):
        c, ctx = C[s:s+BATCH_SIZE], CTX[s:s+BATCH_SIZE]
        cur_lr = LR * max(1e-4, 1 - (epoch * len(C) + s) / (len(C) * EPOCHS))

        vc, vo = Win[c], Wout[ctx]
        p = 1 / (1 + np.exp(-np.clip((vc * vo).sum(1), -20, 20)))
        g = (p - 1)[:, None]
        loss += -np.log(p + 1e-9).sum()
        np.add.at(Win,  c,   -cur_lr * g * vo)
        np.add.at(Wout, ctx, -cur_lr * g * vc)

        ns = np.random.randint(0, V, (len(c), NEG_SAMPLES))
        for k in range(NEG_SAMPLES):
            n = ns[:, k]; vn = Wout[n]
            pn = 1 / (1 + np.exp(-np.clip((Win[c] * vn).sum(1), -20, 20)))
            gn = pn[:, None]
            loss += -np.log(1 - pn + 1e-9).sum()
            np.add.at(Win,  c, -cur_lr * gn * vn)
            np.add.at(Wout, n, -cur_lr * gn * Win[c])

    print(f"Epoch {epoch+1}/{EPOCHS} — loss={loss/len(C):.4f}")

Epoch 1/5 — loss=2.0637
Epoch 2/5 — loss=1.3227
Epoch 3/5 — loss=1.1597
Epoch 4/5 — loss=1.0792
Epoch 5/5 — loss=1.0391


## 5. Save Embeddings as JSON

In [13]:
norms = np.linalg.norm(Win, axis=1, keepdims=True) + 1e-9
Win_norm = Win / norms

embeddings = {token: Win_norm[i].tolist() for i, token in enumerate(idx2token)}

with open(OUTPUT_PATH, "w") as f:
    json.dump(embeddings, f)

print(f"Saved {len(embeddings)} embeddings to {OUTPUT_PATH}")

Saved 37640 embeddings to ./data//embeddings/genre_embeddings.json
